In [4]:
!pip install transformers scikit-learn pandas

In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel
import json

In [6]:
# ---------- LOAD DATA ----------
def load_data(path):
    df = pd.read_csv(path, sep="\t", header=None)
    df.columns = ["text", "labels", "id"]
    return df

train_df = load_data("/content/train.tsv")
dev_df   = load_data("/content/dev.tsv")
test_df  = load_data("/content/test.tsv")


# ---------- LOAD LABEL NAMES ----------
with open("/content/emotions.txt") as f:
    emotions = [line.strip() for line in f.readlines()]

label2emotion = {i: e for i, e in enumerate(emotions)}


# ---------- LOAD EKMAN MAPPING ----------
with open("/content/ekman_mapping.json") as f:
    ekman_map = json.load(f)

# 🔥 FIX: reverse mapping
inverse_ekman = {}
for ekman_label, emo_list in ekman_map.items():
    for emo in emo_list:
        inverse_ekman[emo] = ekman_label

In [7]:
# ---------- DEFINE TARGET LABELS ----------
ekman_classes = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]


# ---------- PARSE LABELS (FIXED FOR YOUR DATA) ----------
def parse_labels(x):
    try:
        return [int(x)]   # single-label → list
    except:
        return []

In [8]:
# ---------- MAP TO EKMAN ----------
def map_to_ekman(label_list):
    out = set()

    for l in label_list:
        emo = label2emotion.get(l, None)
        if emo in inverse_ekman:
            out.add(inverse_ekman[emo])

    return list(out)


# ---------- MULTI-LABEL VECTOR ----------
def to_vector(labels):
    vec = np.zeros(len(ekman_classes))
    for l in labels:
        if l in ekman_classes:
            vec[ekman_classes.index(l)] = 1
    return vec

In [9]:
# ---------- PROCESS FUNCTION ----------
def process_df(df):
    df = df.copy()

    df["label_list"] = df["labels"].apply(parse_labels)
    df["ekman"] = df["label_list"].apply(map_to_ekman)
    df["final_labels"] = df["ekman"].apply(to_vector)

    # remove rows with no Ekman mapping
    df = df[df["final_labels"].apply(lambda x: x.sum() > 0)]

    return df

In [10]:
# ---------- APPLY ----------
train_df = process_df(train_df)
dev_df   = process_df(dev_df)
test_df  = process_df(test_df)

In [11]:
print(train_df.columns)

Index(['text', 'labels', 'id', 'label_list', 'ekman', 'final_labels'], dtype='object')


In [12]:
print(train_df["ekman"].head(10))

2        [anger]
3         [fear]
4        [anger]
5     [surprise]
6          [joy]
8          [joy]
10    [surprise]
13         [joy]
14       [anger]
16         [joy]
Name: ekman, dtype: object


In [13]:
print(set(sum(train_df["ekman"].tolist(), [])))

{'disgust', 'anger', 'sadness', 'surprise', 'joy', 'fear'}


In [14]:
from collections import Counter

def build_vocab(texts, max_vocab=20000):
    counter = Counter()

    for text in texts:
        tokens = text.lower().split()
        counter.update(tokens)

    vocab = {word: i+2 for i, (word, _) in enumerate(counter.most_common(max_vocab))}

    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1

    return vocab

vocab = build_vocab(train_df["text"])

In [15]:
print(len(vocab))

20002


In [16]:
from torch.utils.data import Dataset

class LSTMDataset(Dataset):
    def __init__(self, df, vocab, max_len=64):
        self.texts = df["text"].tolist()
        self.labels = df["final_labels"].tolist()
        self.vocab = vocab
        self.max_len = max_len

    def encode(self, text):
        tokens = text.lower().split()
        ids = [self.vocab.get(t, 1) for t in tokens][:self.max_len]
        ids += [0] * (self.max_len - len(ids))
        return ids

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.encode(self.texts[idx])),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float32)
        }

In [17]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 128)
        self.lstm = nn.LSTM(128, 128, batch_first=True)
        self.fc = nn.Linear(128, 6)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])  # ❌ NO sigmoid here

In [18]:
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
labels = np.array(train_df["final_labels"].tolist())

pos_weights = (len(labels) - labels.sum(axis=0)) / (labels.sum(axis=0) + 1e-6)

pos_weights = torch.tensor(pos_weights, dtype=torch.float32).to(device)

print("Class weights:", pos_weights)

Class weights: tensor([ 5.0560, 46.1586, 44.6019,  0.8177, 10.0726,  5.6099], device='cuda:0')


In [19]:
import torch.nn as nn
from sklearn.metrics import f1_score

def train_epoch(model, loader, optimizer, device, criterion):
    model.train()
    total_loss = 0

    for batch in loader:
        labels = batch["labels"].to(device)

        if "attention_mask" in batch:
            outputs = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device)
            )
        else:
            outputs = model(batch["input_ids"].to(device))

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()
    preds, true = [], []

    with torch.no_grad():
        for batch in loader:
            labels = batch["labels"].cpu().numpy()

            if "attention_mask" in batch:
                outputs = model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device)
                )
            else:
                outputs = model(batch["input_ids"].to(device))

            outputs = torch.sigmoid(outputs).cpu().numpy()  # ✅ here
            pred = (outputs > 0.5).astype(int)

            preds.extend(pred)
            true.extend(labels)

    micro = f1_score(true, preds, average="micro")
    macro = f1_score(true, preds, average="macro")

    return micro, macro

In [20]:
from torch.utils.data import DataLoader

# LSTM
lstm_train_loader = DataLoader(LSTMDataset(train_df, vocab), batch_size=32, shuffle=True)
lstm_dev_loader   = DataLoader(LSTMDataset(dev_df, vocab), batch_size=32)
lstm_test_loader  = DataLoader(LSTMDataset(test_df, vocab), batch_size=32)


In [21]:
lstm_model = LSTMModel(len(vocab)).to(device)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

for epoch in range(8):
    loss = train_epoch(lstm_model, lstm_train_loader, optimizer, device, criterion)
    micro, macro = evaluate(lstm_model, lstm_dev_loader, device)

    print(f"LSTM Epoch {epoch+1}")
    print("Loss:", loss)
    print("Dev Micro F1:", micro)
    print("Dev Macro F1:", macro)

LSTM Epoch 1
Loss: 1.157474921412299
Dev Micro F1: 0.564276048714479
Dev Macro F1: 0.12024221453287197
LSTM Epoch 2
Loss: 1.1558244216994304
Dev Micro F1: 0.19638024357239514
Dev Macro F1: 0.11487043795967378
LSTM Epoch 3
Loss: 1.1556032274009747
Dev Micro F1: 0.4742895805142084
Dev Macro F1: 0.16300246224741044
LSTM Epoch 4
Loss: 1.1556550120625249
Dev Micro F1: 0.3910690121786198
Dev Macro F1: 0.12752216158780247
LSTM Epoch 5
Loss: 1.155546438547831
Dev Micro F1: 0.11299052774018944
Dev Macro F1: 0.050040194769469
LSTM Epoch 6
Loss: 1.155447019543245
Dev Micro F1: 0.3910690121786198
Dev Macro F1: 0.12752216158780247
LSTM Epoch 7
Loss: 1.1554039871010533
Dev Micro F1: 0.3668809201623816
Dev Macro F1: 0.17028240930234095
LSTM Epoch 8
Loss: 1.1554002578316982
Dev Micro F1: 0.3668809201623816
Dev Macro F1: 0.17028240930234095


In [23]:
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
import torch

def evaluate(model, loader, device):
    model.eval()
    preds, true = [], []

    with torch.no_grad():
        for batch in loader:
            labels = batch["labels"].cpu().numpy()

            # forward pass (LSTM or BERT compatible)
            if "attention_mask" in batch:
                outputs = model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device)
                )
            else:
                outputs = model(batch["input_ids"].to(device))

            # logits → probabilities
            outputs = torch.sigmoid(outputs).cpu().numpy()

            # threshold
            pred = (outputs > 0.5).astype(int)

            preds.extend(pred)
            true.extend(labels)

    preds = np.array(preds)
    true = np.array(true)

    # F1 scores
    micro = f1_score(true, preds, average="micro")
    macro = f1_score(true, preds, average="macro")

    # Subset accuracy (exact match)
    subset_acc = accuracy_score(true, preds)

    # Element-wise accuracy
    element_acc = (preds == true).mean()

    return micro, macro, subset_acc, element_acc

In [24]:
micro, macro, subset_acc, element_acc = evaluate(lstm_model, lstm_dev_loader, device)

print(f"LSTM Epoch {epoch+1}")
print("Loss:", loss)
print("Micro F1:", micro)
print("Macro F1:", macro)
print("Subset Acc:", subset_acc)
print("Element Acc:", element_acc)

LSTM Epoch 8
Loss: 1.1554002578316982
Micro F1: 0.3668809201623816
Macro F1: 0.17028240930234095
Subset Acc: 0.0
Element Acc: 0.5779206134415877


In [26]:
import pandas as pd

# create a single-row dataframe from current values
df = pd.DataFrame([{
    "epoch": epoch + 1,
    "loss": loss,
    "micro_f1": micro,
    "macro_f1": macro,
    "subset_accuracy": subset_acc,
    "element_accuracy": element_acc
}])

# append to CSV (does NOT overwrite)
df.to_csv("current_training_results.csv", mode="a", header=not pd.io.common.file_exists("lstm_trainset_results.csv"), index=False)

In [25]:
import os
import torch

save_path = "./lstm_model"
os.makedirs(save_path, exist_ok=True)

torch.save({
    "model_state_dict": lstm_model.state_dict(),
    "vocab": vocab,
    "ekman_classes": ekman_classes
}, os.path.join(save_path, "lstm_model.pt"))

print("LSTM model saved.")

LSTM model saved.


In [27]:
import pandas as pd

micro, macro, subset_acc, element_acc = evaluate(lstm_model, lstm_test_loader, device)

df = pd.DataFrame([{
    "model": "LSTM",
    "micro_f1": micro,
    "macro_f1": macro,
    "subset_accuracy": subset_acc,
    "element_accuracy": element_acc
}])

df.to_csv("lstm_test_results.csv", index=False)

print("Test results saved.")

Test results saved.
